# SQLite VERSION

In [14]:
import os
import re
import hashlib
import sqlite3
import tkinter as tk
from tkinter import filedialog, messagebox, simpledialog
from docx import Document
from PyPDF2 import PdfReader
from pptx import Presentation

In [15]:
DB_FILE = "Database\dataset.db"

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\ACER\AppData\Local\Temp\ipykernel_7720\2048713905.py:1: SyntaxWarning: invalid escape sequence '\d'
  DB_FILE = "Database\dataset.db"


## DATABASE

In [16]:
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS datasets (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    id_teks INTEGER,
    sumber_teks TEXT,
    id_kalimat INTEGER,
    kalimat TEXT,
    hash_teks TEXT
)
""")
conn.commit()


In [17]:
last_uploaded_data = []

## UPLOAD FILE

In [18]:
def extract_text(file_path):
    ext = file_path.split('.')[-1].lower()
    try:
        if ext == "txt":
            with open(file_path, 'r', encoding='utf-8') as f:
                return f.read()
        elif ext == "docx":
            doc = Document(file_path)
            return "\n".join([p.text for p in doc.paragraphs])
        elif ext == "pdf":
            reader = PdfReader(file_path)
            text = ""
            for page in reader.pages:
                if page.extract_text():
                    text += page.extract_text() + "\n"
            return text
        elif ext == "pptx":
            prs = Presentation(file_path)
            text = ""
            for slide in prs.slides:
                for shape in slide.shapes:
                    if hasattr(shape, "text"):
                        text += shape.text + "\n"
            return text
        else:
            return ""
    except:
        return ""

In [19]:
def clean_text(text):
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,?!\"\'-]', '', text)
    return text.strip()

def split_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if len(s.strip()) > 5]

def generate_hash(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

In [20]:
def get_existing_hashes():
    cursor.execute("SELECT DISTINCT hash_teks FROM datasets")
    return {row[0] for row in cursor.fetchall()}

def get_last_id_teks():
    cursor.execute("SELECT MAX(id_teks) FROM datasets")
    result = cursor.fetchone()[0]
    return result if result else 0

def get_total_files():
    cursor.execute("SELECT COUNT(DISTINCT id_teks) FROM datasets")
    return cursor.fetchone()[0]

def get_total_sentences():
    cursor.execute("SELECT COUNT(*) FROM datasets")
    result = cursor.fetchone()[0]
    return result if result else 0

In [21]:
def upload_files():
    global last_uploaded_data
    
    file_paths = filedialog.askopenfilenames(
        filetypes=[("Supported Files", "*.txt *.docx *.pdf *.pptx")]
    )
    
    if not file_paths:
        return
    
    existing_hashes = get_existing_hashes()
    last_id = get_last_id_teks()
    
    added = 0
    skipped = 0
    last_uploaded_data = []
    
    for file_path in file_paths:
        raw = extract_text(file_path)
        cleaned = clean_text(raw)
        if not cleaned:
            continue
        
        file_hash = generate_hash(cleaned)
        
        if file_hash in existing_hashes:
            skipped += 1
            continue
        
        last_id += 1
        sentences = split_sentences(cleaned)
        
        for i, kalimat in enumerate(sentences, start=1):
            cursor.execute("""
                INSERT INTO datasets (id_teks, sumber_teks, id_kalimat, kalimat, hash_teks)
                VALUES (?, ?, ?, ?, ?)
            """, (last_id, os.path.basename(file_path), i, kalimat, file_hash))
            
            last_uploaded_data.append(
                (last_id, os.path.basename(file_path), i, kalimat, file_hash)
            )
        
        added += 1
        existing_hashes.add(file_hash)
        root.update_idletasks()
    
    conn.commit()
    update_info_label()
    
    messagebox.showinfo(
        "Selesai",
        f"File ditambahkan: {added}\nFile duplikat dilewati: {skipped}"
    )


## DELETE FILE

In [22]:
def delete_dataset():
    cursor.execute("SELECT DISTINCT id_teks, sumber_teks FROM datasets")
    rows = cursor.fetchall()
    
    if not rows:
        messagebox.showwarning("Warning", "Database kosong.")
        return
    
    file_list = "\n".join([f"{r[0]} - {r[1]}" for r in rows])
    
    selected = simpledialog.askinteger(
        "Hapus Dataset",
        f"Pilih ID yang ingin dihapus:\n\n{file_list}"
    )
    
    if selected is None:
        return
    
    cursor.execute("DELETE FROM datasets WHERE id_teks=?", (selected,))
    conn.commit()
    
    update_info_label()
    messagebox.showinfo("Sukses", "Dataset berhasil dihapus.")

## EXPORT FILE

In [23]:
def export_last_upload():
    if not last_uploaded_data:
        messagebox.showwarning("Warning", "Belum ada upload baru.")
        return
    
    save_path = filedialog.asksaveasfilename(
        defaultextension=".csv",
        filetypes=[("CSV file", "*.csv")]
    )
    
    if save_path:
        import csv
        with open(save_path, "w", newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(["id_teks","sumber_teks","id_kalimat","kalimat","hash_teks"])
            writer.writerows(last_uploaded_data)
        
        messagebox.showinfo("Sukses", "Upload terakhir berhasil diekspor.")

def export_all_dataset():
    cursor.execute("""
        SELECT id_teks, sumber_teks, id_kalimat, kalimat, hash_teks
        FROM datasets
        ORDER BY id_teks, id_kalimat
    """)
    
    rows = cursor.fetchall()
    
    if not rows:
        messagebox.showwarning("Warning", "Database kosong.")
        return
    
    save_path = filedialog.asksaveasfilename(
        defaultextension=".csv",
        filetypes=[("CSV file", "*.csv")],
        title="Simpan Seluruh Dataset"
    )
    
    if save_path:
        import csv
        with open(save_path, "w", newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(["id_teks","sumber_teks","id_kalimat","kalimat","hash_teks"])
            writer.writerows(rows)
        
        messagebox.showinfo("Sukses", "Seluruh dataset berhasil diekspor.")

In [24]:
def update_info_label():
    total_files = get_total_files()
    total_sentences = get_total_sentences()
    
    info_label.config(
        text=f"Total File Unik: {total_files} | Total Seluruh Kalimat: {total_sentences}"
    )

In [25]:
def safe_exit():
    conn.close()
    root.after(100, root.destroy)

## GUI

In [26]:
root = tk.Tk()
root.title("Dataset Manager - Hendra Ahmad Yani")
root.geometry("520x450")
root.resizable(False, False)

tk.Label(root, text="MESIN DATASET",
         font=("Arial",14,"bold")).pack(pady=15)

info_label = tk.Label(root, font=("Arial",11))
info_label.pack(pady=5)

update_info_label()

tk.Button(root, text="Upload File", width=35,
          command=upload_files).pack(pady=10)

tk.Button(root, text="Hapus Dataset Berdasarkan ID", width=35,
          command=delete_dataset).pack(pady=5)

tk.Button(root, text="Export Upload Terakhir", width=35,
          command=export_last_upload).pack(pady=5)

tk.Button(root, text="Export Seluruh Dataset", width=35,
          command=export_all_dataset).pack(pady=5)

tk.Button(root, text="Keluar", width=35,
          command=safe_exit).pack(pady=20)

root.protocol("WM_DELETE_WINDOW", safe_exit)

root.mainloop()